# Data Cleaning
*Team p1zz@: Sean Qin, Daniel Cai, Surya Venkataraman, Anjeli Noémi (Mimi)*

Cleans the September 2026 Divvy trips and Meteostat weather and builds the tables each analysis uses. Run top to bottom; outputs go to `data/processed/`:

- `trips_clean.csv`: kept rides
- `exclusions.csv`: rides removed by each rule
- `station_events.csv`: one row per departure or arrival at a known station
- `station_day.csv`: morning and later flows, recovery, day type, and weather per station and day
- `station_day_by_rider.csv`: the same flows split by member/casual
- `stations.csv`: name and coordinates of each station
- `weather_daily.csv`: daily temperature and precipitation at Midway

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_TRIPS = Path("../data/raw/202609-divvy-tripdata.csv")
RAW_WEATHER = Path("../data/raw/meteostat_daily_2026_midway_72534.csv.gz")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

# Time windows
MORNING = (6, 10)    # 6:00-9:59 a.m.
LATER = (10, 20)     # 10:00 a.m.-7:59 p.m.
LABOR_DAY = 7        # Monday, Sept 7, 2026

# Duration cutoffs
MIN_DURATION_SEC = 60          # false starts / re-docks
MAX_DURATION_SEC = 24 * 3600   # lost or stolen bikes

## Load the raw data

In [2]:
raw = pd.read_csv(RAW_TRIPS)
print(raw.shape)
raw.head()

(770562, 13)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,B3C076B2D6BAC911,electric_bike,2026-09-01 19:28:37.087,2026-09-01 19:33:03.047,NaN,NaN,NaN,NaN,41.87,-87.67,41.87,-87.66,member
1,468834E54297FB3F,electric_bike,2026-09-24 20:38:01.486,2026-09-24 20:44:54.382,NaN,NaN,NaN,NaN,41.91,-87.70,41.90,-87.71,member
2,276E6265CF1F0238,electric_bike,2026-09-24 20:01:27.138,2026-09-24 20:08:32.366,NaN,NaN,NaN,NaN,41.90,-87.71,41.91,-87.70,member
3,F297F75928C557B7,electric_bike,2026-09-29 21:20:03.900,2026-09-29 21:42:29.179,NaN,NaN,NaN,NaN,41.96,-87.68,41.92,-87.72,member
4,0C21D327353C6D73,electric_bike,2026-09-30 13:23:43.088,2026-09-30 13:27:10.920,NaN,NaN,NaN,NaN,41.79,-87.60,41.79,-87.59,member


In [3]:
overview = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(2),
    "unique": raw.nunique(),
})
overview

,dtype,missing,missing_%,unique
ride_id,object,0,0.00,770562
rideable_type,object,0,0.00,2
started_at,object,0,0.00,770363
ended_at,object,0,0.00,770226
start_station_name,object,160341,20.81,1554
start_station_id,object,160341,20.81,1554
end_station_name,object,170033,22.07,1574
end_station_id,object,170033,22.07,1574
start_lat,float64,0,0.00,1591
start_lng,float64,0,0.00,1586


The four station columns are the only ones with meaningful missingness: about 21% of start stations and 22% of end stations.
The 644 missing end coordinates all belong to rides that are also missing an end station.

## Timestamps and day type

Timestamps are Chicago local time. September 2026 has no daylight-saving change, so no time-zone adjustment is needed.

In [4]:
trips = raw.copy()
trips["started_at"] = pd.to_datetime(trips["started_at"], format="mixed")
trips["ended_at"] = pd.to_datetime(trips["ended_at"], format="mixed")
trips["duration_sec"] = (trips["ended_at"] - trips["started_at"]).dt.total_seconds()

trips["start_month"] = trips["started_at"].dt.month
trips["start_day"] = trips["started_at"].dt.day
trips["start_hour"] = trips["started_at"].dt.hour
trips["end_month"] = trips["ended_at"].dt.month
trips["end_day"] = trips["ended_at"].dt.day
trips["end_hour"] = trips["ended_at"].dt.hour

print("started_at:", trips["started_at"].min(), "to", trips["started_at"].max())
print("ended_at:  ", trips["ended_at"].min(), "to", trips["ended_at"].max())
trips["start_month"].value_counts()

started_at: 2026-08-31 00:53:20.571000 to 2026-09-30 23:59:38.090000
ended_at:   2026-09-01 00:00:18.076000 to 2026-09-30 23:59:48.043000


start_month
9    770511
8        51
Name: count, dtype: int64

Divvy files rides under the month they end, so 51 rides start on Aug 31. Only September events are used for the station-day table, so these rides count as arrivals but not departures.

In [5]:
# One row per day of the study month
calendar = pd.DataFrame({"year": 2026, "month": 9, "day": np.arange(1, 31)})
calendar["date"] = pd.to_datetime(calendar[["year", "month", "day"]])
calendar["weekday"] = calendar["date"].dt.day_name()
calendar["day_type"] = np.select(
    [calendar["day"] == LABOR_DAY, calendar["date"].dt.dayofweek >= 5],
    ["Labor Day", "Weekend"],
    default="Weekday",
)
calendar = calendar.drop(columns=["year", "month"])
calendar["day_type"].value_counts()

day_type
Weekday      21
Weekend       8
Labor Day     1
Name: count, dtype: int64

## Duplicates

In [6]:
print("duplicate ride_id:", trips["ride_id"].duplicated().sum())
print("fully duplicated rows:", raw.duplicated().sum())

# Near-duplicates: same bike type, rider type, stations, and start/end time to the second.
# The first 19 characters of the raw timestamp text are 'YYYY-MM-DD HH:MM:SS'.
trips["start_sec"] = raw["started_at"].str[:19]
trips["end_sec"] = raw["ended_at"].str[:19]
near_key = ["rideable_type", "member_casual", "start_station_id", "end_station_id", "start_sec", "end_sec"]
known_both = trips["start_station_id"].notna() & trips["end_station_id"].notna()

in_group = known_both & trips.duplicated(subset=near_key, keep=False)
trips["flag_near_duplicate"] = known_both & trips.duplicated(subset=near_key, keep="first")
print("rows in a near-duplicate group:", in_group.sum(), "| second copies flagged:", trips["flag_near_duplicate"].sum())
trips[in_group].sort_values("start_sec")[["ride_id"] + near_key]

duplicate ride_id: 0


fully duplicated rows: 0


rows in a near-duplicate group: 8 | second copies flagged: 4


,ride_id,rideable_type,member_casual,start_station_id,end_station_id,start_sec,end_sec
73564,6E71B764604F8252,electric_bike,casual,CHI00336,CHI02366,2026-09-06 11:04:44,2026-09-06 11:34:52
522083,91B648886B53F218,electric_bike,casual,CHI00336,CHI02366,2026-09-06 11:04:44,2026-09-06 11:34:52
217866,A57FA8A23A1EB5A6,classic_bike,casual,CHI00313,CHI00253,2026-09-10 14:42:02,2026-09-10 14:51:18
220871,2C1F28CECAC632CB,classic_bike,casual,CHI00313,CHI00253,2026-09-10 14:42:02,2026-09-10 14:51:18
207336,533A11D301780959,electric_bike,member,CHI00242,CHI01772,2026-09-18 18:05:31,2026-09-18 18:15:53
207388,2B37F5A509EC2E23,electric_bike,member,CHI00242,CHI01772,2026-09-18 18:05:31,2026-09-18 18:15:53
387868,59297AB5C6D1D2B7,electric_bike,casual,CHI02383,CHI01747,2026-09-19 18:55:08,2026-09-19 19:04:06
387884,12FF1A92D6555DB9,electric_bike,casual,CHI02383,CHI01747,2026-09-19 18:55:08,2026-09-19 19:04:06


There are no repeated ride IDs. Four pairs of rides match on every field down to the second; we keep the first ride in each pair and flag the second as a duplicate.

## Trip durations

In [7]:
trips["flag_too_short"] = trips["duration_sec"] < MIN_DURATION_SEC
trips["flag_too_long"] = trips["duration_sec"] > MAX_DURATION_SEC

same_station = trips["start_station_id"].notna() & (trips["start_station_id"] == trips["end_station_id"])
print(f"< {MIN_DURATION_SEC}s: {trips['flag_too_short'].sum():,}"
      f"  (same start and end station: {(trips['flag_too_short'] & same_station).sum():,})")
print(f"> 24h: {trips['flag_too_long'].sum():,}"
      f"  (no end station: {(trips['flag_too_long'] & trips['end_station_id'].isna()).sum():,})")
trips["duration_sec"].describe(percentiles=[.01, .25, .5, .75, .99])

< 60s: 18,610  (same start and end station: 6,349)


> 24h: 633  (no end station: 622)


count    770562.000000
mean        909.488887
std        2917.616594
min           0.070000
1%           17.244610
25%         328.981250
50%         564.786500
75%         976.197000
99%        4840.697840
max       90066.163000
Name: duration_sec, dtype: float64

Rides under 60 s are false starts or re-docks under Divvy's own rule; about a third end where they started. Of the 633 rides over 24 h, 622 have no end station, so they are most likely lost or stolen bikes. Both groups are dropped (the cutoffs are set in Setup).

## Rider categories

In [8]:
print(trips["member_casual"].value_counts(dropna=False), "\n")
print(trips["rideable_type"].value_counts(dropna=False), "\n")
pd.crosstab(trips["member_casual"], trips["rideable_type"], margins=True)

member_casual
member    476442
casual    294120
Name: count, dtype: int64 

rideable_type
electric_bike    580638
classic_bike     189924
Name: count, dtype: int64 



rideable_type,classic_bike,electric_bike,All
member_casual,,,
casual,59677,234443,294120
member,130247,346195,476442
All,189924,580638,770562


Both category columns are complete and have only the expected values, so no recoding is needed.

## Station IDs

In [9]:
# Each ID should map to exactly one name, and each name to one ID
for side in ["start", "end"]:
    ids, names = f"{side}_station_id", f"{side}_station_name"
    names_per_id = trips.groupby(ids)[names].nunique()
    ids_per_name = trips.groupby(names)[ids].nunique()
    print(f"{side}: {trips[ids].nunique():,} ids | ids with >1 name: {(names_per_id > 1).sum()}"
          f" | names with >1 id: {(ids_per_name > 1).sum()}")

trips["start_missing"] = trips["start_station_id"].isna()
trips["end_missing"] = trips["end_station_id"].isna()
coverage = trips.groupby("rideable_type").agg(
    rides=("ride_id", "count"),
    start_missing_pct=("start_missing", "mean"),
    end_missing_pct=("end_missing", "mean"),
)
coverage[["start_missing_pct", "end_missing_pct"]] = (coverage[["start_missing_pct", "end_missing_pct"]] * 100).round(1)
coverage

start: 1,554 ids | ids with >1 name: 0 | names with >1 id: 0
end: 1,574 ids | ids with >1 name: 0 | names with >1 id: 0


,rides,start_missing_pct,end_missing_pct
rideable_type,,,
classic_bike,189924,0.0,0.3
electric_bike,580638,27.6,29.2


Station IDs and names match one-to-one, so the ID is used as the station key.

All missing stations are e-bikes, which can be parked away from a dock. Flows only count known stations, so e-bikes are undercounted compared with classic bikes.

## Apply exclusions

In [10]:
rules = [
    ("Near-duplicate (second copy)", "flag_near_duplicate"),
    (f"Duration < {MIN_DURATION_SEC} s", "flag_too_short"),
    ("Duration > 24 h", "flag_too_long"),
]
keep = pd.Series(True, index=trips.index)
log = [{"step": "Raw rides", "removed": 0, "remaining": len(trips)}]
for label, col in rules:
    removed = (keep & trips[col]).sum()
    keep = keep & ~trips[col]
    log.append({"step": label, "removed": removed, "remaining": keep.sum()})
exclusions = pd.DataFrame(log)

drop_cols = ["start_sec", "end_sec", "start_missing", "end_missing",
             "flag_near_duplicate", "flag_too_short", "flag_too_long"]
trips_clean = trips[keep].drop(columns=drop_cols)
trips_clean["has_start_station"] = trips_clean["start_station_id"].notna()
trips_clean["has_end_station"] = trips_clean["end_station_id"].notna()
trips_clean["has_both_stations"] = trips_clean["has_start_station"] & trips_clean["has_end_station"]
exclusions

,step,removed,remaining
0,Raw rides,0,770562
1,Near-duplicate (second copy),4,770558
2,Duration < 60 s,18610,751948
3,Duration > 24 h,633,751315


In [11]:
print(f"Kept {len(trips_clean):,} of {len(trips):,} rides ({len(trips_clean)/len(trips):.1%})")
print(f"  with start station: {trips_clean['has_start_station'].mean():.1%}")
print(f"  with end station:   {trips_clean['has_end_station'].mean():.1%}")
print(f"  with both (usable for routes): {trips_clean['has_both_stations'].mean():.1%}")

Kept 751,315 of 770,562 rides (97.5%)
  with start station: 79.9%
  with end station:   78.9%
  with both (usable for routes): 66.6%


## Station-day table

Each ride gives a departure at its start station and an arrival at its end station. Using September events at known stations, we count flows per station and day:

- Morning net departures = departures − arrivals, 6:00–9:59 a.m.
- Later net arrivals = arrivals − departures, 10:00 a.m.–7:59 p.m.
- Recovery = later net arrivals ÷ morning net departures, only when morning net departures > 0. Recovery ≥ 1 is full recovery.

In [12]:
keep_cols = ["ride_id", "member_casual", "rideable_type"]
departures = trips_clean[trips_clean["has_start_station"]][
    keep_cols + ["start_station_id", "start_station_name", "started_at", "start_month", "start_day", "start_hour"]
].rename(columns={"start_station_id": "station_id", "start_station_name": "station_name", "started_at": "time",
                  "start_month": "month", "start_day": "day", "start_hour": "hour"})
departures["event"] = "departure"

arrivals = trips_clean[trips_clean["has_end_station"]][
    keep_cols + ["end_station_id", "end_station_name", "ended_at", "end_month", "end_day", "end_hour"]
].rename(columns={"end_station_id": "station_id", "end_station_name": "station_name", "ended_at": "time",
                  "end_month": "month", "end_day": "day", "end_hour": "hour"})
arrivals["event"] = "arrival"

events = pd.concat([departures, arrivals], ignore_index=True)
assert len(events) == len(departures) + len(arrivals)
print(f"{len(departures):,} departures + {len(arrivals):,} arrivals = {len(events):,} events")
print("events outside September (dropped):", (events["month"] != 9).sum())
events = events[events["month"] == 9].drop(columns="month")
events["window"] = np.select(
    [events["hour"].between(MORNING[0], MORNING[1] - 1), events["hour"].between(LATER[0], LATER[1] - 1)],
    ["morning", "later"],
    default="other",
)
events["event"].value_counts()

600,193 departures + 592,929 arrivals = 1,193,122 events
events outside September (dropped): 42


event
departure    600151
arrival      592929
Name: count, dtype: int64

In [13]:
def station_day_flows(ev, index):
    """Count morning/later departures and arrivals per row of `index` and compute recovery."""
    ev = ev[ev["window"] != "other"].copy()
    ev["flow"] = ev["window"] + "_" + ev["event"] + "s"
    flows = ev.pivot_table(index=index, columns="flow", values="ride_id", aggfunc="count")
    flows = flows.fillna(0).astype(int).reset_index()
    flows.columns.name = None

    flows["morning_net_departures"] = flows["morning_departures"] - flows["morning_arrivals"]
    flows["later_net_arrivals"] = flows["later_arrivals"] - flows["later_departures"]
    flows["has_morning_deficit"] = flows["morning_net_departures"] > 0
    flows["recovery"] = np.where(flows["has_morning_deficit"],
                                 flows["later_net_arrivals"] / flows["morning_net_departures"], np.nan)
    flows["full_recovery"] = flows["recovery"] >= 1
    return flows

station_names = events.drop_duplicates("station_id")[["station_id", "station_name"]]
station_day = station_day_flows(events, index=["station_id", "day"])

# One row per station-day, and every event is counted
assert not station_day.duplicated(subset=["station_id", "day"]).any()
assert station_day[["morning_departures", "morning_arrivals", "later_departures", "later_arrivals"]].sum().sum() \
    == (events["window"] != "other").sum()

# Merges should keep the row count
n_rows = len(station_day)
station_day = station_day.merge(station_names, on="station_id", how="left", validate="many_to_one", indicator=True)
print("station names:", station_day["_merge"].value_counts().to_dict())
station_day = station_day.drop(columns="_merge").merge(calendar, on="day", how="left", validate="many_to_one", indicator=True)
print("calendar:     ", station_day["_merge"].value_counts().to_dict())
station_day = station_day.drop(columns="_merge")
assert len(station_day) == n_rows
print(station_day.shape)
station_day.sort_values("morning_net_departures", ascending=False).head()

station names: {'both': 30383, 'left_only': 0, 'right_only': 0}
calendar:      {'both': 30383, 'left_only': 0, 'right_only': 0}
(30383, 15)


,station_id,day,later_arrivals,later_departures,morning_arrivals,morning_departures,morning_net_departures,later_net_arrivals,has_morning_deficit,recovery,full_recovery,station_name,date,weekday,day_type
1941,CHI00281,23,59,36,13,46,33,23,True,0.696970,False,Clark St & Elm St,2026-09-23,Wednesday,Weekday
10867,CHI00604,29,31,17,3,29,26,14,True,0.538462,False,Wabash Ave & 16th St,2026-09-29,Tuesday,Weekday
6792,CHI00450,24,43,31,1,26,25,12,True,0.480000,False,Indiana Ave & Roosevelt Rd,2026-09-24,Thursday,Weekday
1929,CHI00281,11,45,37,20,45,25,8,True,0.320000,False,Clark St & Elm St,2026-09-11,Friday,Weekday
1927,CHI00281,9,58,54,8,32,24,4,True,0.166667,False,Clark St & Elm St,2026-09-09,Wednesday,Weekday


`station_day` has one row for every station and day with at least one departure or arrival between 6 a.m. and 8 p.m.
A station-day with no activity cannot have a morning deficit, so leaving those days out does not change any recovery result.

In [14]:
# Same flows split by member/casual
station_day_by_rider = station_day_flows(events, index=["station_id", "day", "member_casual"])
station_day_by_rider = station_day_by_rider.merge(calendar, on="day", how="left", validate="many_to_one")
station_day_by_rider.head()

,station_id,day,member_casual,later_arrivals,later_departures,morning_arrivals,morning_departures,morning_net_departures,later_net_arrivals,has_morning_deficit,recovery,full_recovery,date,weekday,day_type
0,CHI00005,1,casual,3,5,1,0,-1,-2,False,NaN,False,2026-09-01,Tuesday,Weekday
1,CHI00005,1,member,17,9,0,5,5,8,True,1.6,True,2026-09-01,Tuesday,Weekday
2,CHI00005,2,casual,5,7,0,1,1,-2,True,-2.0,False,2026-09-02,Wednesday,Weekday
3,CHI00005,2,member,17,14,2,3,1,3,True,3.0,True,2026-09-02,Wednesday,Weekday
4,CHI00005,3,casual,2,3,0,1,1,-1,True,-1.0,False,2026-09-03,Thursday,Weekday


In [15]:
# Station locations: each station has one fixed coordinate, so take the median over all rides
starts = trips_clean[["start_station_id", "start_lat", "start_lng"]].set_axis(["station_id", "lat", "lng"], axis=1)
ends = trips_clean[["end_station_id", "end_lat", "end_lng"]].set_axis(["station_id", "lat", "lng"], axis=1)
coords = pd.concat([starts, ends]).dropna().groupby("station_id")[["lat", "lng"]].median().reset_index()
stations = station_names.merge(coords, on="station_id", how="left", validate="one_to_one")
print(stations.shape, "| missing coordinates:", stations["lat"].isna().sum())
stations.head()

(1589, 4) | missing coordinates: 0


,station_id,station_name,lat,lng
0,CHI00307,California Ave & Milwaukee Ave,41.922695,-87.697153
1,CHI01183,Wood St & Chicago Ave,41.895634,-87.672069
2,CHI00261,Morgan Ave & 14th Pl,41.862378,-87.651062
3,CHI00214,Campbell Ave & Montrose Ave,41.961524,-87.691177
4,CHI00340,California Ave & Division St,41.903029,-87.697474


## Weather

Meteostat daily data (`https://data.meteostat.net/daily/2026/{station}.csv.gz`). O'Hare (72530) has no September precipitation, so we use Chicago Midway (72534). Its temperature and precipitation come from observed reports (`metar`, `ghcnd`), not model estimates.

In [16]:
wx = pd.read_csv(RAW_WEATHER)
wx = wx[wx["month"] == 9]
print("days:", len(wx), "| missing prcp:", wx["prcp"].isna().sum(), "| missing temp:", wx["temp"].isna().sum())
for col in ["temp_source", "tmax_source", "prcp_source"]:
    print(col, wx[col].unique().tolist())

weather = wx[["day", "temp", "tmin", "tmax", "prcp"]].rename(columns={
    "temp": "temp_mean_c", "tmin": "temp_min_c", "tmax": "temp_max_c", "prcp": "prcp_mm",
})
# Keep missing precipitation as NaN, not 0
weather["rainy"] = np.where(weather["prcp_mm"].isna(), np.nan, weather["prcp_mm"] > 0)
weather

days: 30 | missing prcp: 0 | missing temp: 0
temp_source ['metar']
tmax_source ['ghcnd']
prcp_source ['ghcnd']


,day,temp_mean_c,temp_min_c,temp_max_c,prcp_mm,rainy
243,1,29.5,25.0,34.4,0.0,0.0
244,2,30.2,25.6,35.6,0.0,0.0
245,3,29.0,26.1,33.3,0.0,0.0
246,4,29.4,25.6,33.9,0.0,0.0
247,5,25.7,22.8,28.3,0.0,0.0
248,6,23.5,21.7,26.1,0.0,0.0
249,7,23.0,18.9,26.7,0.0,0.0
250,8,24.3,18.9,30.6,0.0,0.0
251,9,24.2,21.7,26.7,1.0,1.0
252,10,22.0,18.3,24.4,0.0,0.0


In [17]:
n_sd, n_rider = len(station_day), len(station_day_by_rider)
station_day = station_day.merge(weather, on="day", how="left", validate="many_to_one", indicator=True)
print("weather match:", station_day["_merge"].value_counts().to_dict())
station_day = station_day.drop(columns="_merge")
station_day_by_rider = station_day_by_rider.merge(weather, on="day", how="left", validate="many_to_one")
assert len(station_day) == n_sd and len(station_day_by_rider) == n_rider
print("rainy days:", int(weather["rainy"].sum()), "of", len(weather))

weather match: {'both': 30383, 'left_only': 0, 'right_only': 0}
rainy days: 7 of 30


## Checks

In [18]:
# Member and casual flows add up to the station-day totals
flow_cols = ["morning_departures", "morning_arrivals", "later_departures", "later_arrivals"]
rider_sum = station_day_by_rider.groupby(["station_id", "day"])[flow_cols].sum().reset_index()
check = station_day[["station_id", "day"] + flow_cols].merge(
    rider_sum, on=["station_id", "day"], how="outer", suffixes=("", "_riders"), indicator=True, validate="one_to_one")
assert (check["_merge"] == "both").all()
for c in flow_cols:
    assert check[c].equals(check[c + "_riders"])

# Recovery is defined exactly on deficit days, and every day has weather
assert station_day.loc[~station_day["has_morning_deficit"], "recovery"].isna().all()
assert station_day.loc[station_day["has_morning_deficit"], "recovery"].notna().all()
assert station_day["temp_max_c"].notna().all()
assert stations["lat"].notna().all()
print("All checks passed.")

All checks passed.


## Save outputs

In [19]:
trips_clean.to_csv(OUT / "trips_clean.csv", index=False)
exclusions.to_csv(OUT / "exclusions.csv", index=False)
events.to_csv(OUT / "station_events.csv", index=False)
station_day.to_csv(OUT / "station_day.csv", index=False)
station_day_by_rider.to_csv(OUT / "station_day_by_rider.csv", index=False)
weather.to_csv(OUT / "weather_daily.csv", index=False)
stations.to_csv(OUT / "stations.csv", index=False)

for f in sorted(OUT.glob("*.csv")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:7.1f} MB")

exclusions.csv                   0.0 MB
station_day.csv                  3.6 MB
station_day_by_rider.csv         5.1 MB
station_events.csv             138.9 MB
stations.csv                     0.1 MB
trips_clean.csv                166.7 MB
weather_daily.csv                0.0 MB
